# ML Pro · Обмеження LLM

Обмеження великих мовних моделей: токени, контекстне вікно, «пам'ять», пошук у базі знань (RAG) і temperature.

## 0. Підготовка

Встановлення модуля:

`!pip install -q tiktoken`

In [ ]:
!pip show tiktoken

Name: tiktoken
Version: 0.14.0
Summary: tiktoken is a fast BPE tokeniser for use with OpenAI's models
Home-page: https://github.com/openai/tiktoken
Author: Shantanu Jain
Author-email: shantanu@openai.com
License: MIT License

Copyright (c) 2022 OpenAI, Shantanu Jain

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICUL

In [ ]:
import tiktoken
import numpy as np
import pandas as pd
print("Готово! Можна працювати.")

Готово! Можна працювати.


## Завдання 1. Токени: як модель «бачить» текст

Модель читає не літери і не слова, а **токени** — шматочки тексту. Порахуймо їх бібліотекою `tiktoken` від OpenAI.

In [ ]:
enc = tiktoken.get_encoding("o200k_base")    # токенізатор моделей GPT-4o і новіших
type(enc)

tiktoken.core.Encoding

In [ ]:
enc = tiktoken.get_encoding("o200k_base")    # токенізатор моделей GPT-4o і новіших

def tokens_word(n: int) -> str:
  """1 токен, 2 токени, 5 токенів - правильне закінчення"""
  if n % 10 == 1 and n % 100 != 11:
    return "токен"
  if n % 10 in (2, 3, 4) and n % 100 in (12, 13, 14):
    return "токени"
  return "токенів"

def show_tokens(text: str, encoder: tiktoken.core.Encoding=enc) -> int:
  ids = encoder.encode(text)
  pieces = [encoder.decode([i]) for i in ids]
  print(f"{len(text)} символів -> {len(ids)} {tokens_word(len(ids))}")
  # print(ids)
  # print(" | ".join(pieces))

  display(pd.DataFrame({
      "pieces" : pieces,
      "ids": ids
  }).T.style.hide(axis="index").hide(axis="columns"))

  return len(ids)

In [ ]:
text_en = "Large language models predict the next token."
text_uk = "Великі мовні моделі передбачають наступний токен."

n_en = show_tokens(text_en)
print()
n_uk = show_tokens(text_uk)
print()
print(f"Український текст займає в {n_uk / n_en:.1f} більше токенів")

45 символів -> 8 токенів


Large,language,models,predict,the,next,token,.
42565,6439,7015,17946,290,2613,6602,13



49 символів -> 15 токенів


В,елик,і,мов,ні,мод,елі,перед,бач,ають,наступ,ний,ток,ен,.
3540,79934,732,185190,6774,22389,63165,20897,143901,176261,66753,12979,75096,776,13



Український текст займає в 1.9 більше токенів


> Український текст «дорожчий» для того самого змісту (англійський чи український).  
Потрібно більше токенів, тож він дорожчий через API і швидше заповнює контекстне вікно.

In [ ]:
show_tokens("Hello World! qwertyuiop[]\';lkjhjhgfdss")

38 символів -> 16 токенів


Hello,World,!,q,wert,y,ui,op,[],';,lk,j,hj,hg,fd,ss
13225,5922,0,2335,21773,88,1866,467,1951,15290,31466,73,98719,115323,13073,1087


16

Порівняння **старого** токенізатору (GPT-3.5 і GPT-4, 2023) з **новим**.

In [ ]:
old = tiktoken.get_encoding("cl100k_base")  # токенізатор GPT-3.5 / GPT-4

for name, e in [("cl100k_base (2023)", old), ("o200k_base (2024+)", enc)]:
    en, uk = len(e.encode(text_en)), len(e.encode(text_uk))
    print(f"{name}: англійська {en}, українська {uk} токенів -> різниця x{uk / en:.1f}")

cl100k_base (2023): англійська 8, українська 26 токенів -> різниця x3.2
o200k_base (2024+): англійська 8, українська 15 токенів -> різниця x1.9


> Новий токенізатор має більший «словник» (≈200 тис. токенів замість ≈100 тис.) і краще знає українські шматки слів, тому різниця з англійською менша.

А тепер — чому моделі довго помилялися в задачі підрахунку літер — «скільки літер r у слові strawberry».

In [ ]:
show_tokens("strawberry")
show_tokens("полуниця")
print()
print("Python рахує літери точно:", "strawberry".count("r"))
print("Python перевертає слово:", "полуниця"[::-1])

10 символів -> 3 токенів


st,raw,berry
302,1618,19772


8 символів -> 4 токенів


пол,у,ниц,я
67478,442,28366,544



Python рахує літери точно: 3
Python перевертає слово: яцинулоп


> Модель отримує токени (наприклад, `str|aw|berry`), а не окремі літери, і «бачить» слово шматками. Python працює з кожним символом окремо. Тому сучасні чат-боти для точних задач запускають код.